<a href="https://colab.research.google.com/github/FVALENCIAR3/An-lisis-de-CDR-s-automatizado/blob/main/Modelo_an%C3%A1lisis_CDR_con_IA_Mejora_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import io
import re
from google.colab import files
from IPython.display import clear_output

print("📂 Sube tus archivos CSV o Excel (Soporta múltiples operadores al mismo tiempo):")
uploaded = files.upload()

df_list = []

def limpiar_fechas(s):
    s = s.astype(str).str.strip()
    mask_14 = s.str.match(r'^\d{14}$')
    fechas_14 = pd.to_datetime(s[mask_14], format='%Y%m%d%H%M%S', errors='coerce')
    fechas_otros = pd.to_datetime(s[~mask_14], format='mixed', dayfirst=True, errors='coerce')
    return pd.concat([fechas_14, fechas_otros])

# FÓRMULA DE CONVERSIÓN: HEXADECIMAL A DECIMAL (Ej: Movistar)
def normalizar_celda(val):
    val = str(val).strip().upper()
    if val.endswith('.0'):
        val = val[:-2]
    # Detecta el formato hexadecimal con guión (Ej: 08B2F7-09, 7D17-0E18, B101-1035)
    if re.match(r'^[0-9A-F]{4,6}-[0-9A-F]{2,4}$', val):
        try:
            parts = val.split('-')
            # Convierte cada parte de Base 16 (Hex) a Base 10 (Dec)
            dec_parts = [str(int(p, 16)) for p in parts]
            return '-'.join(dec_parts)
        except:
            return val
    return val

for file_name, file_content in uploaded.items():
    try:
        if file_name.endswith('.csv'):
            try:
                df = pd.read_csv(io.BytesIO(file_content), sep=';', encoding='utf-8-sig', low_memory=False)
                if len(df.columns) <= 1:
                    df = pd.read_csv(io.BytesIO(file_content), sep=',', encoding='utf-8-sig', low_memory=False)
            except:
                df = pd.read_csv(io.BytesIO(file_content), sep=None, engine='python')
        else:
            df = pd.read_excel(io.BytesIO(file_content))

        df.columns = df.columns.str.strip().str.lower()

        # Identificar columnas dinámicamente
        col_tel = next((c for c in ['numero', 'numero_origen', 'numero_que_marca', 'numero_que_navega', 'originador'] if c in df.columns), None)
        col_fec = next((c for c in ['fecha_hora_inicio', 'fecha_trafico', 'fecha_hora_inicio_llamada', 'fecha_hora_inicio_sesion', 'fecha_hora'] if c in df.columns), None)
        col_lat = next((c for c in ['latitud', 'latitud_n'] if c in df.columns), None)
        col_lon = next((c for c in ['longitud', 'longitud_w'] if c in df.columns), None)
        col_cel = next((c for c in ['celda_decimal', 'cell_id_voz', 'celda', 'celda_inicio_llamada', 'bts_id', 'celda_hex'] if c in df.columns), None)

        df_temp = pd.DataFrame()

        if col_tel:
            df_temp['numero_limpio'] = df[col_tel].astype(str).str.replace(r'\D', '', regex=True).str[-10:]
        if col_fec:
            df_temp['fecha_limpia'] = limpiar_fechas(df[col_fec])
        if col_lat and col_lon:
            df_temp['latitud'] = pd.to_numeric(df[col_lat].astype(str).str.replace(',', '.'), errors='coerce')
            df_temp['longitud'] = pd.to_numeric(df[col_lon].astype(str).str.replace(',', '.'), errors='coerce')
        if col_cel:
            df_temp['celda'] = df[col_cel].apply(normalizar_celda)

        # Almacenar todo el registro puro para exportaciones posteriores
        df_temp['registro_original'] = df.to_dict('records')
        df_list.append(df_temp)
        print(f"✅ Procesado: {file_name}")

    except Exception as e:
        print(f"❌ Error procesando {file_name}: {e}")

if df_list:
    df_master = pd.concat(df_list, ignore_index=True)
    clear_output()
    print("✅ ¡Archivos consolidados! Celdas Hexadecimales convertidas y números a 10 dígitos.")
    print(f"Total registros: {len(df_master)}")
else:
    print("No se encontraron datos.")

✅ ¡Archivos consolidados! Celdas Hexadecimales convertidas y números a 10 dígitos.
Total registros: 150676


In [3]:
import folium
from folium.plugins import AntPath
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd

if 'df_master' not in globals():
    print("⚠️ Primero ejecuta la Celda 1 para cargar los datos.")
else:
    try:
        fecha_min = df_master['fecha_limpia'].min().date()
        fecha_max = df_master['fecha_limpia'].max().date()
    except:
        fecha_min = fecha_max = None

    display(widgets.HTML("<h3>📍 Búsqueda Geográfica Individual</h3>"))

    text_buscar = widgets.Text(value='', placeholder='Ej: 3157658841', description='Número:')
    dropdown_top = widgets.Dropdown(options=['Todos', 'Top 10', 'Top 5', 'Top 3', 'Top 1'], value='Todos', description='Mostrar:')

    dp_inicio = widgets.DatePicker(description='Fecha Ini:', value=fecha_min, layout=widgets.Layout(width='200px'))
    tm_inicio = widgets.Text(value='00:00', description='Hora Ini:', layout=widgets.Layout(width='150px'))
    dp_fin = widgets.DatePicker(description='Fecha Fin:', value=fecha_max, layout=widgets.Layout(width='200px'))
    tm_fin = widgets.Text(value='23:59', description='Hora Fin:', layout=widgets.Layout(width='150px'))

    btn_mapa = widgets.Button(description=' Trazar Mapa', button_style='success', icon='map-marker')
    salida_mapa = widgets.Output()

    display(widgets.HBox([text_buscar, dropdown_top]))
    display(widgets.HBox([dp_inicio, tm_inicio, dp_fin, tm_fin]))
    display(btn_mapa)
    display(salida_mapa)

    def generar_mapa(b):
        with salida_mapa:
            clear_output()
            numero_buscado = text_buscar.value.strip()

            if len(numero_buscado) != 10:
                print("⚠️ Ingresa un número de exactamente 10 dígitos.")
                return

            df_usuario = df_master[(df_master['numero_limpio'] == numero_buscado) & (df_master['latitud'].notna())].copy()

            try:
                dt_ini = pd.to_datetime(f"{dp_inicio.value} {tm_inicio.value}")
                dt_fin = pd.to_datetime(f"{dp_fin.value} {tm_fin.value}")
                df_usuario = df_usuario[(df_usuario['fecha_limpia'] >= dt_ini) & (df_usuario['fecha_limpia'] <= dt_fin)]
            except:
                pass

            if df_usuario.empty:
                print(f"❌ No hay historial de coordenadas para {numero_buscado} en este rango de tiempo.")
                return

            df_usuario = df_usuario.sort_values(by='fecha_limpia')

            df_agrupado = df_usuario.groupby(['latitud', 'longitud']).agg(
                visitas=('numero_limpio', 'count'),
                primera_visita=('fecha_limpia', 'min'),
                ultima_visita=('fecha_limpia', 'max'),
                celdas=('celda', lambda x: ', '.join(x.dropna().unique().astype(str)))
            ).reset_index().sort_values(by='visitas', ascending=False).reset_index(drop=True)

            # --- REDACCIÓN DE LA SINOPSIS ---
            tot_conexiones = len(df_usuario)
            top_lugar = df_agrupado.iloc[0]['celdas']
            top_visitas = df_agrupado.iloc[0]['visitas']

            sinopsis = f"""
            <div style="background-color:#e8f4f8; padding:15px; border-left: 5px solid #5bc0de; border-radius:5px; margin-bottom: 15px;">
                <h4 style="margin-top:0; color:#31708f;">💡 Sinopsis del Comportamiento (Patrón de Vida)</h4>
                <p>El objetivo <b>{numero_buscado}</b> registra una actividad total de <b>{tot_conexiones}</b> conexiones geo-posicionadas dentro del periodo consultado. Su zona de mayor confort, residencia o trabajo habitual corresponde a la cobertura de la celda <b>{top_lugar}</b>, lugar donde acumuló el mayor número de impactos (<b>{top_visitas}</b> visitas registradas).</p>
            </div>
            """
            display(HTML(sinopsis))

            # --- MAPA ---
            n = len(df_agrupado) if dropdown_top.value == 'Todos' else int(dropdown_top.value.replace('Top ', ''))
            df_mostrar = df_agrupado.head(n)

            mapa = folium.Map(location=[df_mostrar.iloc[0]['latitud'], df_mostrar.iloc[0]['longitud']], zoom_start=14)

            for i, row in df_mostrar.iterrows():
                coord = [row['latitud'], row['longitud']]
                ranking = i + 1
                color = 'red' if ranking == 1 else 'orange' if ranking <=3 else 'blue'
                popup_html = f"""<div style="min-width: 200px;"><b>Rank:</b> #{ranking}<br><b>Visitas:</b> {row['visitas']}<br><b>Celda:</b> {row['celdas']}<br><b>Inicio:</b> {row['primera_visita']}<br><b>Fin:</b> {row['ultima_visita']}</div>"""
                folium.CircleMarker(location=coord, radius=min(20, 8 + row['visitas']), color=color, fill=True).add_to(mapa)
                folium.Marker(location=coord, popup=folium.Popup(popup_html, max_width=300), tooltip=f"Rank #{ranking}", icon=folium.Icon(color=color)).add_to(mapa)

            if dropdown_top.value == 'Todos' and len(df_usuario) > 1:
                AntPath(locations=df_usuario[['latitud', 'longitud']].values.tolist(), delay=1000, color='purple', weight=4).add_to(mapa)

            display(mapa)

    btn_mapa.on_click(generar_mapa)

HTML(value='<h3>📍 Búsqueda Geográfica Individual</h3>')

Button(button_style='success', description=' Trazar Mapa', icon='map-marker', style=ButtonStyle())

Output()

In [4]:
import pandas as pd
import itertools
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import gc
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import zipfile
import os

if 'df_master' not in globals():
    print("⚠️ Ejecuta la Celda 1 primero para cargar los datos.")
else:
    try:
        fecha_min = df_master['fecha_limpia'].min().date()
        fecha_max = df_master['fecha_limpia'].max().date()
    except:
        fecha_min = fecha_max = None

    chk_colombia = widgets.Checkbox(value=True, description='Omitir Plataformas (Solo Móviles Colombia que inician con 3)', layout=widgets.Layout(width='500px'))
    dp_inicio = widgets.DatePicker(description='Fecha Ini:', value=fecha_min, layout=widgets.Layout(width='200px'))
    tm_inicio = widgets.Text(value='00:00', description='Hora Ini:', layout=widgets.Layout(width='150px'))
    dp_fin = widgets.DatePicker(description='Fecha Fin:', value=fecha_max, layout=widgets.Layout(width='200px'))
    tm_fin = widgets.Text(value='23:59', description='Hora Fin:', layout=widgets.Layout(width='150px'))
    txt_excluir = widgets.Text(description='Excluir Núm:', placeholder='Ej: 3151234567', layout=widgets.Layout(width='350px'))
    btn_ejecutar = widgets.Button(description=' Generar Análisis', button_style='primary', icon='play')

    out_resultados = widgets.Output()

    display(HTML("<h3>⚙️ Panel de Filtros para Análisis</h3>"))
    display(chk_colombia)
    display(widgets.HBox([dp_inicio, tm_inicio, dp_fin, tm_fin]))
    display(widgets.HBox([txt_excluir, btn_ejecutar]))
    display(out_resultados)

    def procesar_analisis(b):
        with out_resultados:
            clear_output()
            print("⏳ Analizando registros...")

            df_analisis = df_master.dropna(subset=['numero_limpio', 'celda', 'fecha_limpia']).copy()

            if chk_colombia.value:
                df_analisis = df_analisis[df_analisis['numero_limpio'].str.match(r'^3\d{9}$')]

            try:
                dt_ini = pd.to_datetime(f"{dp_inicio.value} {tm_inicio.value}")
                dt_fin = pd.to_datetime(f"{dp_fin.value} {tm_fin.value}")
                df_analisis = df_analisis[(df_analisis['fecha_limpia'] >= dt_ini) & (df_analisis['fecha_limpia'] <= dt_fin)]
            except: pass

            nums_excluir = [x.strip() for x in txt_excluir.value.split(',') if x.strip()]
            if nums_excluir: df_analisis = df_analisis[~df_analisis['numero_limpio'].isin(nums_excluir)]

            if df_analisis.empty:
                print("⚠️ No hay datos para analizar con los filtros aplicados.")
                return

            df_top_celdas = df_analisis.groupby(['celda', 'numero_limpio']).size().reset_index(name='conexiones')
            top_10_celdas = df_top_celdas.sort_values(by='conexiones', ascending=False).head(10).reset_index(drop=True)

            df_analisis['ventana'] = df_analisis['fecha_limpia'].dt.floor('15min')
            agrupado_tiempo = df_analisis.groupby(['celda', 'ventana'])['numero_limpio'].unique()
            conteo_tiempo = Counter()
            for numeros in agrupado_tiempo:
                if 1 < len(numeros) <= 150: conteo_tiempo.update(itertools.combinations(sorted(numeros), 2))
            df_encuentros = pd.DataFrame([{'Número A': p[0], 'Número B': p[1], 'Coincidencias': c} for p, c in conteo_tiempo.most_common(10)])

            # --- SINOPSIS ANALÍTICA ---
            display(HTML("<hr><h2>📊 INFORME DE INTELIGENCIA DE SEÑALES</h2>"))
            sinopsis = """<div style="background-color:#e8f4f8; padding:15px; border-left: 5px solid #5bc0de; border-radius:5px;">
                            <h4 style="margin-top:0; color:#31708f;">💡 Sinopsis Analítica de la Red</h4>"""

            if not top_10_celdas.empty:
                t_c = top_10_celdas.iloc[0]
                sinopsis += f"<p><b>📌 Zonas Calientes:</b> El mayor volumen de tráfico aislado fue generado por el número <b>{t_c['numero_limpio']}</b> anclado a la celda <b>{t_c['celda']}</b>, sumando un total de <b>{t_c['conexiones']}</b> conexiones. Esto indica un punto de permanencia alto.</p>"
            if not df_encuentros.empty:
                t_e = df_encuentros.iloc[0]
                sinopsis += f"<p><b>🤝 Co-Ubicación Confirmada:</b> Se detectó un patrón de encuentro de alta probabilidad temporal (rango de 15 min) entre las líneas <b>{t_e['Número A']}</b> y <b>{t_e['Número B']}</b>. Estos dos objetivos compartieron el mismo espacio físico de celda en <b>{t_e['Coincidencias']}</b> ocasiones distintas.</p>"
            else:
                sinopsis += "<p><b>🤝 Co-Ubicación:</b> No se evidenciaron patrones de encuentro simultáneo en la franja temporal de 15 min.</p>"

            sinopsis += "</div><hr>"
            display(HTML(sinopsis))

            display(HTML("<h3 style='color:#0275d8;'>🏆 Top 10: Frecuencia de Números por Celda</h3>"))
            if not top_10_celdas.empty: display(top_10_celdas.style.background_gradient(cmap='Blues'))

            display(HTML("<h3 style='color:#d9534f;'>📍 Top 10: Encuentros Probables (Espacio/Tiempo)</h3>"))
            if not df_encuentros.empty: display(df_encuentros.style.background_gradient(cmap='Oranges'))

            # --- EXPORTACIONES ---
            display(HTML("<hr><h3>⬇️ Exportar Registros Originales, Mapa y Gráfica (ZIP)</h3>"))
            op_c = [f"{r['celda']} | {r['numero_limpio']}" for _, r in top_10_celdas.iterrows()] if not top_10_celdas.empty else []
            op_e = [f"{r['Número A']} | {r['Número B']}" for _, r in df_encuentros.iterrows()] if not df_encuentros.empty else []

            drop_celda = widgets.Dropdown(options=op_c, description='Frecuentes:', layout=widgets.Layout(width='400px'))
            btn_celda = widgets.Button(description='Descargar ZIP', button_style='info')
            drop_encuentro = widgets.Dropdown(options=op_e, description='Encuentros:', layout=widgets.Layout(width='400px'))
            btn_encuentro = widgets.Button(description='Descargar ZIP', button_style='warning')
            out_export = widgets.Output()

            def generar_exportables(df_datos, titulo_base):
                df_excel = pd.DataFrame(df_datos['registro_original'].tolist()).astype(str)
                for col in df_excel.columns: df_excel[col] = df_excel[col].str.replace(r'\.0$', '', regex=True).replace(['nan', 'NaT', 'None'], '')
                archivo_excel = f"{titulo_base}.xlsx"
                df_excel.to_excel(archivo_excel, index=False)

                archivo_img = f"{titulo_base}.png"
                plt.figure(figsize=(10, 4))
                df_datos['fecha_hora'] = pd.to_datetime(df_datos['fecha_limpia'])
                sns.histplot(data=df_datos, x='fecha_hora', bins=20, kde=True, color='purple')
                plt.title("Línea de Tiempo de Conexiones")
                plt.tight_layout()
                plt.savefig(archivo_img)
                plt.close()

                archivo_mapa = f"{titulo_base}_Mapa.html"
                df_mapa = df_datos.dropna(subset=['latitud', 'longitud'])
                if not df_mapa.empty:
                    m = folium.Map(location=[df_mapa['latitud'].mean(), df_mapa['longitud'].mean()], zoom_start=14)
                    for _, r in df_mapa.iterrows():
                        folium.CircleMarker([r['latitud'], r['longitud']], radius=5, color='red', fill=True, popup=str(r['fecha_limpia'])).add_to(m)
                    m.save(archivo_mapa)
                else:
                    with open(archivo_mapa, 'w') as f: f.write("<h3>Sin coordenadas</h3>")

                archivo_zip = f"{titulo_base}_Exportacion.zip"
                with zipfile.ZipFile(archivo_zip, 'w') as zf:
                    zf.write(archivo_excel); zf.write(archivo_img); zf.write(archivo_mapa)

                files.download(archivo_zip)
                print(f"✅ Paquete ZIP generado exitosamente.")

            def exp_celda_click(b):
                with out_export:
                    clear_output()
                    celda, num = drop_celda.value.split(' | ')
                    df_c = df_analisis[(df_analisis['celda'] == celda) & (df_analisis['numero_limpio'] == num)]
                    generar_exportables(df_c, f"Historial_{num}_Celda_{celda}")

            def exp_enc_click(b):
                with out_export:
                    clear_output()
                    nA, nB = drop_encuentro.value.split(' | ')
                    dA = df_analisis[df_analisis['numero_limpio'] == nA][['celda', 'ventana']]
                    dB = df_analisis[df_analisis['numero_limpio'] == nB][['celda', 'ventana']]
                    inters = pd.merge(dA, dB, on=['celda', 'ventana']).drop_duplicates()
                    df_cruz = pd.merge(df_analisis, inters, on=['celda', 'ventana'])
                    df_cruz = df_cruz[df_cruz['numero_limpio'].isin([nA, nB])]
                    generar_exportables(df_cruz, f"Encuentro_{nA}_vs_{nB}")

            btn_celda.on_click(exp_celda_click)
            btn_encuentro.on_click(exp_enc_click)
            display(widgets.HBox([drop_celda, btn_celda]))
            display(widgets.HBox([drop_encuentro, btn_encuentro]))
            display(out_export)

            del agrupado_tiempo, df_top_celdas
            gc.collect()

    btn_ejecutar.on_click(procesar_analisis)

Checkbox(value=True, description='Omitir Plataformas (Solo Móviles Colombia que inician con 3)', layout=Layout…

Output()

In [5]:
# 1. Instalar librerías para leer PDF y Word automáticamente
!pip install PyPDF2 python-docx -q

import pandas as pd
import re
import folium
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import io
import PyPDF2
import docx

if 'df_master' not in globals():
    print("⚠️ Ejecuta la Celda 1 primero para cargar los datos maestros.")
else:
    # =========================================================================
    # INTERFAZ DE ENTRADA (TEXTO + ARCHIVOS)
    # =========================================================================
    display(HTML("<h2>📝 Redacción de Hechos y Lectura de Documentos</h2>"))
    display(HTML("<p>Escribe el contexto del caso o <b>sube los oficios/informes en PDF, Word o TXT</b>. El sistema leerá el documento, extraerá los <b>números de 10 dígitos</b> automáticamente y cruzará la evidencia.</p>"))

    txt_hechos = widgets.Textarea(
        value='',
        placeholder='Puedes escribir un resumen aquí o dejarlo en blanco si vas a subir un documento...',
        description='Hechos:',
        layout=widgets.Layout(width='95%', height='100px')
    )

    # Nuevo widget para subir documentos (.txt, .pdf, .docx)
    upload_docs = widgets.FileUpload(
        accept='.txt, .pdf, .docx',
        multiple=True,
        description='📁 Subir Docs'
    )

    btn_informe = widgets.Button(
        description=' Generar Informe Analítico',
        button_style='primary',
        icon='file-text',
        layout=widgets.Layout(width='250px')
    )

    out_informe = widgets.Output()

    # Mostrar la interfaz
    display(txt_hechos)
    display(widgets.HBox([upload_docs, btn_informe]))
    display(out_informe)

    def generar_informe(b):
        with out_informe:
            clear_output()
            print("⏳ Procesando información y documentos...")

            # 1. Obtener texto del cuadro de texto
            texto_completo = txt_hechos.value.strip() + "\n"

            # 2. Extraer texto de los archivos subidos (PDF, Word, TXT)
            if upload_docs.value:
                # Compatibilidad con diferentes versiones de ipywidgets
                archivos = upload_docs.value if isinstance(upload_docs.value, (list, tuple)) else upload_docs.value.values()

                for file_info in archivos:
                    nombre_arch = file_info['name']
                    contenido = file_info['content']

                    try:
                        if nombre_arch.endswith('.txt'):
                            texto_completo += contenido.decode('utf-8', errors='ignore') + "\n"
                        elif nombre_arch.endswith('.docx'):
                            doc = docx.Document(io.BytesIO(contenido))
                            for para in doc.paragraphs:
                                texto_completo += para.text + "\n"
                        elif nombre_arch.endswith('.pdf'):
                            lector = PyPDF2.PdfReader(io.BytesIO(contenido))
                            for pagina in lector.pages:
                                txt_pag = pagina.extract_text()
                                if txt_pag:
                                    texto_completo += txt_pag + "\n"
                        print(f"✅ Documento leído con éxito: {nombre_arch}")
                    except Exception as e:
                        print(f"❌ Error leyendo {nombre_arch}: {e}")

            if not texto_completo.strip():
                print("⚠️ No ingresaste texto ni subiste documentos válidos.")
                return

            # 3. Extracción de Inteligencia (Números de 10 dígitos)
            numeros_implicados = list(set(re.findall(r'\b\d{10}\b', texto_completo)))

            display(HTML("<hr><h2>📊 INFORME ANALÍTICO DE INTELIGENCIA</h2>"))

            if not numeros_implicados:
                display(HTML("<div style='background-color:#fcf8e3; padding:10px; border-left: 5px solid #faebcc;'><b>Aviso:</b> No se detectaron números de celular de 10 dígitos ni en el texto ni en los documentos adjuntos.</div>"))
                return

            # 4. Cruce con la Base de Datos Maestra
            df_caso = df_master[df_master['numero_limpio'].isin(numeros_implicados)].copy()

            if df_caso.empty:
                display(HTML(f"<div style='background-color:#f2dede; padding:10px; border-left: 5px solid #ebccd1;'><b>Resultado:</b> Los números hallados en el texto ({', '.join(numeros_implicados)}) <b>NO</b> registran tráfico en tu evidencia (Celda 1).</div>"))
                return

            # 5. Procesamiento de Datos para la Sinopsis
            total_registros = len(df_caso)
            nums_encontrados = df_caso['numero_limpio'].unique()

            if 'fecha_limpia' in df_caso.columns:
                df_caso = df_caso.dropna(subset=['fecha_limpia']).sort_values('fecha_limpia')
                fecha_ini = df_caso['fecha_limpia'].min()
                fecha_fin = df_caso['fecha_limpia'].max()
                lapso = f"Desde el <b>{fecha_ini}</b> hasta el <b>{fecha_fin}</b>"
            else:
                lapso = "No hay datos de fecha disponibles en la evidencia."

            top_celdas = df_caso['celda'].value_counts().head(3).index.tolist()
            celdas_txt = ", ".join([str(c) for c in top_celdas])

            # Resumir el texto para no colapsar la pantalla si subieron un PDF de 100 hojas
            resumen_txt = texto_completo[:400].replace('\n', ' ') + ("..." if len(texto_completo) > 400 else "")

            # 6. REDACCIÓN DE LA SINOPSIS (TIEMPO, MODO Y LUGAR)
            sinopsis_html = f"""
            <div style="background-color:#f9f9f9; padding:15px; border-radius:8px; border: 1px solid #ddd;">
                <h3 style="margin-top:0; color:#333;">📋 Sinopsis Forense Transversal</h3>
                <p style="font-size: 0.9em; color: #555;"><b>Contexto Procesado:</b> <i>"{resumen_txt}"</i></p>

                <h4 style="color:#0275d8;">🕒 TIEMPO</h4>
                <p>El análisis de la evidencia digital abarca un espectro temporal activo {lapso}. Durante este periodo, los objetivos generaron un total de <b>{total_registros}</b> registros de conexión (llamadas/datos), estableciendo una línea de tiempo medible de su actividad de red.</p>

                <h4 style="color:#5cb85c;">📍 LUGAR</h4>
                <p>El rastreo geoespacial y de infraestructura revela que los implicados concentraron su actividad principalmente en las celdas de cobertura: <b>{celdas_txt}</b>. Estas antenas representan el área de confort y de influencia principal durante la ventana de tiempo analizada.</p>

                <h4 style="color:#d9534f;">⚙️ MODO</h4>
                <p>La lectura de documentos logró perfilar <b>{len(nums_encontrados)}</b> línea(s) telefónica(s) con cruce positivo en la evidencia: <b>{', '.join(nums_encontrados)}</b>. El comportamiento refleja el uso dinámico de la red móvil, interactuando con la infraestructura que se detalla a continuación.</p>
            </div>
            """
            display(HTML(sinopsis_html))

            # 7. VISUALIZACIONES
            display(HTML("<hr><h3>📈 Análisis Visual del Comportamiento</h3>"))

            # --- GRAFICA DE TIEMPO (Línea de Tiempo) ---
            if 'fecha_limpia' in df_caso.columns and not df_caso.empty:
                plt.figure(figsize=(12, 4))
                df_caso['fecha_solo_dia'] = df_caso['fecha_limpia'].dt.date
                conteo_dias = df_caso.groupby(['fecha_solo_dia', 'numero_limpio']).size().reset_index(name='eventos')

                sns.lineplot(data=conteo_dias, x='fecha_solo_dia', y='eventos', hue='numero_limpio', marker='o', linewidth=2)
                plt.title("Frecuencia de Tráfico en el Tiempo (Por Número)", fontweight='bold')
                plt.xlabel("Fecha")
                plt.ylabel("Cantidad de Conexiones (Impactos)")
                plt.grid(True, linestyle='--', alpha=0.7)
                plt.legend(title="Número Implicado", bbox_to_anchor=(1.05, 1), loc='upper left')
                plt.tight_layout()
                plt.show()

            # --- MAPA DE LUGAR ---
            if 'latitud' in df_caso.columns and 'longitud' in df_caso.columns:
                df_mapa = df_caso.dropna(subset=['latitud', 'longitud'])

                if not df_mapa.empty:
                    display(HTML("<h4>Ubicaciones Geoespaciales Frecuentadas por los Implicados</h4>"))

                    lat_centro = df_mapa['latitud'].mean()
                    lon_centro = df_mapa['longitud'].mean()
                    mapa_informe = folium.Map(location=[lat_centro, lon_centro], zoom_start=12)

                    colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'black']
                    color_map = {num: colores[i % len(colores)] for i, num in enumerate(nums_encontrados)}

                    agrupado_mapa = df_mapa.groupby(['numero_limpio', 'latitud', 'longitud']).size().reset_index(name='visitas')

                    for _, row in agrupado_mapa.iterrows():
                        num = row['numero_limpio']
                        color = color_map[num]

                        folium.CircleMarker(
                            location=[row['latitud'], row['longitud']],
                            radius=min(25, 8 + row['visitas']*1.5), # Círculos dinámicos
                            color=color,
                            fill=True,
                            fill_opacity=0.4,
                            tooltip=f"<b>Línea:</b> {num}<br><b>Impactos:</b> {row['visitas']}"
                        ).add_to(mapa_informe)

                    display(mapa_informe)
                else:
                    print("⚠️ Los números extraídos no cuentan con Coordenadas (Lat/Lon) en la evidencia para dibujar el mapa.")

    btn_informe.on_click(generar_informe)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.7 MB/s eta 0:00:00


Textarea(value='', description='Hechos:', layout=Layout(height='100px', width='95%'), placeholder='Puedes escr…

Output()

In [6]:
import pandas as pd
import folium
from folium.plugins import AntPath
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import gc
from google.colab import files
import zipfile

if 'df_master' not in globals():
    print("⚠️ Ejecuta la Celda 1 primero para cargar los datos.")
else:
    try:
        f_min = df_master['fecha_limpia'].min().date()
        f_max = df_master['fecha_limpia'].max().date()
    except:
        f_min = f_max = None

    display(HTML("<h2>🕵️‍♂️ Búsqueda de Victimarios por Co-Desplazamiento (Precisión Avanzada)</h2>"))

    txt_victima = widgets.Text(description='Núm Víctima:', placeholder='Ej: 3157658841', layout=widgets.Layout(width='300px'))
    drop_tiempo = widgets.Dropdown(options=['15min', '30min', '1H', '2H'], value='1H', description='Tolerancia:', layout=widgets.Layout(width='200px'))
    chk_colombia = widgets.Checkbox(value=True, description='Ignorar Plataformas (Solo Móviles Colombia)', layout=widgets.Layout(width='350px'))

    dp_inicio = widgets.DatePicker(description='Fecha Ini:', value=f_min, layout=widgets.Layout(width='200px'))
    tm_inicio = widgets.Text(value='00:00', description='Hora Ini:', layout=widgets.Layout(width='150px'))
    dp_fin = widgets.DatePicker(description='Fecha Fin:', value=f_max, layout=widgets.Layout(width='200px'))
    tm_fin = widgets.Text(value='23:59', description='Hora Fin:', layout=widgets.Layout(width='150px'))

    btn_rastrear = widgets.Button(description=' Buscar Victimarios', button_style='danger', icon='search', layout=widgets.Layout(width='200px'))
    out_rastreo = widgets.Output()

    display(widgets.HBox([txt_victima, drop_tiempo, chk_colombia]))
    display(widgets.HBox([dp_inicio, tm_inicio, dp_fin, tm_fin]))
    display(btn_rastrear)
    display(out_rastreo)

    def rastrear_victimarios(b):
        with out_rastreo:
            clear_output()
            num_victima = txt_victima.value.strip()
            ventana_str = drop_tiempo.value

            if len(num_victima) != 10:
                print("⚠️ Ingresa el número de la víctima a 10 dígitos.")
                return

            print("⏳ Escaneando trayectorias con ventana de tiempo deslizante...")
            df_limpio = df_master.dropna(subset=['numero_limpio', 'celda', 'fecha_limpia']).copy()

            try:
                dt_ini = pd.to_datetime(f"{dp_inicio.value} {tm_inicio.value}")
                dt_fin = pd.to_datetime(f"{dp_fin.value} {tm_fin.value}")
                df_limpio = df_limpio[(df_limpio['fecha_limpia'] >= dt_ini) & (df_limpio['fecha_limpia'] <= dt_fin)]
            except: pass

            df_victima = df_limpio[df_limpio['numero_limpio'] == num_victima].copy()
            if df_victima.empty:
                display(HTML(f"<div style='background-color:#f2dede; padding:10px;'>❌ La víctima ({num_victima}) no presenta tráfico. Revisa los filtros.</div>"))
                return

            df_otros = df_limpio[df_limpio['numero_limpio'] != num_victima].copy()
            if chk_colombia.value:
                df_otros = df_otros[df_otros['numero_limpio'].str.match(r'^3\d{9}$')]

            td_tolerancia = pd.Timedelta(ventana_str)
            resultados = []

            for _, v_row in df_victima.iterrows():
                celda_v = v_row['celda']
                tiempo_v = v_row['fecha_limpia']

                lim_inf = tiempo_v - td_tolerancia
                lim_sup = tiempo_v + td_tolerancia
                match = df_otros[(df_otros['celda'] == celda_v) & (df_otros['fecha_limpia'] >= lim_inf) & (df_otros['fecha_limpia'] <= lim_sup)]

                for _, s_row in match.iterrows():
                    resultados.append({
                        'numero_limpio': s_row['numero_limpio'],
                        'celda': celda_v,
                        'fecha_sospechoso': s_row['fecha_limpia'],
                        'registro_original': s_row['registro_original']
                    })

            intersecciones = pd.DataFrame(resultados)

            if intersecciones.empty:
                display(HTML(f"<div style='background-color:#dff0d8; padding:10px;'>✅ Se analizaron los puntos de la víctima, <b>ningún número</b> la siguió de cerca.</div>"))
                return

            sospechosos = intersecciones.groupby('numero_limpio').agg(
                celdas_compartidas=('celda', 'nunique'), impactos_totales=('celda', 'count')
            ).reset_index().sort_values(by=['celdas_compartidas', 'impactos_totales'], ascending=[False, False]).head(10)

            # --- REDACCIÓN DE LA SINOPSIS ---
            top_sospechoso = sospechosos.iloc[0]
            sinopsis = f"""
            <div style="background-color:#e8f4f8; padding:15px; border-left: 5px solid #d9534f; border-radius:5px; margin-bottom:15px;">
                <h4 style="margin-top:0; color:#c9302c;">💡 Sinopsis de Seguimiento (Co-Traveler Tracker)</h4>
                <p>De acuerdo con la ruta cronológica trazada por la víctima (<b>{num_victima}</b>), el algoritmo perfila al número <b>{top_sospechoso['numero_limpio']}</b> como el <b>Principal Sospechoso / Victimario</b>.</p>
                <p>La evidencia muestra que este objetivo siguió la misma ruta de la víctima saltando a través de <b>{top_sospechoso['celdas_compartidas']}</b> antenas/celdas distintas de la ciudad, impactando <b>{top_sospechoso['impactos_totales']}</b> veces la red, manteniéndose siempre a una diferencia temporal máxima de {ventana_str} respecto a la posición de la víctima.</p>
            </div>
            """
            display(HTML(sinopsis))

            display(HTML("<h3 style='color:#d9534f;'>🎯 Top 10: Posibles Victimarios</h3>"))
            display(sospechosos.style.background_gradient(cmap='Reds', subset=['celdas_compartidas']))

            # --- DESCARGAS ---
            display(HTML("<hr><h4>⬇️ Descargar Evidencia en Paquete ZIP</h4>"))
            drop_sosp = widgets.Dropdown(options=sospechosos['numero_limpio'].tolist(), description='Descargar:')
            btn_desc = widgets.Button(description='Descargar ZIP', button_style='warning')
            out_desc = widgets.Output()

            def descargar_cruce(b):
                with out_desc:
                    clear_output()
                    s_sel = drop_sosp.value

                    cruce_final = intersecciones[intersecciones['numero_limpio'] == s_sel].drop_duplicates(subset=['celda', 'fecha_sospechoso'])
                    df_exp = pd.DataFrame(cruce_final['registro_original'].tolist()).astype(str)
                    for col in df_exp.columns: df_exp[col] = df_exp[col].str.replace(r'\.0$', '', regex=True).replace(['nan', 'NaT', 'None'], '')
                    arch_ex = f"Persecucion_{num_victima}_vs_{s_sel}.xlsx"
                    df_exp.to_excel(arch_ex, index=False)

                    arch_map = f"Rutas_{num_victima}_vs_{s_sel}.html"
                    c_vic = df_victima.dropna(subset=['latitud', 'longitud'])
                    c_sos = df_otros[df_otros['numero_limpio'] == s_sel].dropna(subset=['latitud', 'longitud'])

                    if not c_vic.empty:
                        m = folium.Map(location=[c_vic['latitud'].mean(), c_vic['longitud'].mean()], zoom_start=13)
                        AntPath(locations=c_vic[['latitud', 'longitud']].values.tolist(), color='green', weight=5, tooltip="Víctima").add_to(m)
                        if not c_sos.empty: AntPath(locations=c_sos[['latitud', 'longitud']].values.tolist(), color='red', weight=4, tooltip="Sospechoso").add_to(m)
                        m.save(arch_map)
                    else:
                        with open(arch_map, 'w') as f: f.write("<h3>Sin coordenadas</h3>")

                    arch_zip = f"Evidencia_Persecucion_{s_sel}.zip"
                    with zipfile.ZipFile(arch_zip, 'w') as zf:
                        zf.write(arch_ex); zf.write(arch_map)

                    files.download(arch_zip)
                    print(f"✅ Paquete ZIP descargado.")

            btn_desc.on_click(descargar_cruce)
            display(widgets.HBox([drop_sosp, btn_desc]))
            display(out_desc)

            del df_limpio
            gc.collect()

    btn_rastrear.on_click(rastrear_victimarios)

Button(button_style='danger', description=' Buscar Victimarios', icon='search', layout=Layout(width='200px'), …

Output()

In [7]:
# 1. Instalar librería para crear archivos de Google Earth
!pip install simplekml -q

import pandas as pd
import folium
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import gc
from google.colab import files
import simplekml

if 'df_master' not in globals():
    print("⚠️ Ejecuta la Celda 1 primero para cargar los datos maestros.")
else:
    # Obtener fechas límites
    try:
        f_min = df_master['fecha_limpia'].min().date()
        f_max = df_master['fecha_limpia'].max().date()
    except:
        f_min = f_max = None

    display(HTML("<h2>👤 Perfilamiento de Rutinas y Zonas de Pernocta (Victimario)</h2>"))
    display(HTML("<p>Analiza el comportamiento diario de un objetivo. Descubre dónde duerme (Noche), dónde amanece (Madrugada) y dónde opera (Día). Exportable a Google Earth.</p>"))

    # ---------------------------------------------------------
    # PANEL DE CONTROL
    # ---------------------------------------------------------
    txt_victimario = widgets.Text(description='Victimario:', placeholder='Ej: 3157658841', layout=widgets.Layout(width='300px'))
    drop_top = widgets.Dropdown(options=['Top 10', 'Top 5', 'Top 3', 'Top 1'], value='Top 3', description='Mostrar:', layout=widgets.Layout(width='200px'))
    chk_colombia = widgets.Checkbox(value=True, description='Ignorar Plataformas (Solo Móviles Colombia)', layout=widgets.Layout(width='350px'))

    dp_inicio = widgets.DatePicker(description='Fecha Ini:', value=f_min, layout=widgets.Layout(width='200px'))
    tm_inicio = widgets.Text(value='00:00', description='Hora Ini:', layout=widgets.Layout(width='150px'))
    dp_fin = widgets.DatePicker(description='Fecha Fin:', value=f_max, layout=widgets.Layout(width='200px'))
    tm_fin = widgets.Text(value='23:59', description='Hora Fin:', layout=widgets.Layout(width='150px'))

    btn_analizar = widgets.Button(description=' Generar Perfil de Vida', button_style='primary', icon='user-secret', layout=widgets.Layout(width='220px'))
    out_perfil = widgets.Output()

    # Variable global para guardar los datos listos para el KML
    global df_kml_ready
    df_kml_ready = pd.DataFrame()

    display(widgets.HBox([txt_victimario, drop_top, chk_colombia]))
    display(widgets.HBox([dp_inicio, tm_inicio, dp_fin, tm_fin]))
    display(btn_analizar)
    display(out_perfil)

    def analizar_rutinas(b):
        global df_kml_ready
        with out_perfil:
            clear_output()
            num_vic = txt_victimario.value.strip()

            if len(num_vic) != 10:
                print("⚠️ Ingresa el número del victimario a 10 dígitos.")
                return

            print("⏳ Perfilando comportamientos y franjas horarias...")

            df_limpio = df_master.dropna(subset=['numero_limpio', 'celda', 'fecha_limpia']).copy()

            # Filtro Fechas
            try:
                dt_ini = pd.to_datetime(f"{dp_inicio.value} {tm_inicio.value}")
                dt_fin = pd.to_datetime(f"{dp_fin.value} {tm_fin.value}")
                df_limpio = df_limpio[(df_limpio['fecha_limpia'] >= dt_ini) & (df_limpio['fecha_limpia'] <= dt_fin)]
            except: pass

            # Filtro Móviles
            if chk_colombia.value:
                df_limpio = df_limpio[df_limpio['numero_limpio'].str.match(r'^3\d{9}$')]

            df_obj = df_limpio[df_limpio['numero_limpio'] == num_vic].copy()

            if df_obj.empty:
                display(HTML(f"<div style='background-color:#f2dede; padding:10px;'>❌ <b>Sin registros:</b> El victimario ({num_vic}) no presenta tráfico en los filtros indicados.</div>"))
                return

            # ---------------------------------------------------------
            # CLASIFICACIÓN DE FRANJAS HORARIAS
            # ---------------------------------------------------------
            df_obj['hora'] = df_obj['fecha_limpia'].dt.hour

            def clasificar_horario(h):
                if h >= 22 or h <= 2: return '1. NOCHE (Pernocta: 22:00 - 02:59)'
                elif 3 <= h <= 8: return '2. MAÑANA/MADRUGADA (03:00 - 08:59)'
                else: return '3. DÍA (Actividad: 09:00 - 21:59)'

            df_obj['franja'] = df_obj['hora'].apply(clasificar_horario)

            # Intentar rescatar nombre de la antena si existe
            col_desc = 'descripcion' if 'descripcion' in df_obj.columns else ('direccion' if 'direccion' in df_obj.columns else 'celda')

            # Agrupar por franja y celda
            rutinas = df_obj.groupby(['franja', 'celda']).agg(
                nombre_lugar=(col_desc, 'first'),
                visitas=('celda', 'count'),
                primer_registro=('fecha_limpia', 'min'),
                ultimo_registro=('fecha_limpia', 'max'),
                latitud=('latitud', 'mean'),
                longitud=('longitud', 'mean')
            ).reset_index()

            rutinas = rutinas.sort_values(by=['franja', 'visitas'], ascending=[True, False])

            # Aplicar filtro TOP N por cada franja
            n_top = int(drop_top.value.replace('Top ', ''))
            top_rutinas = rutinas.groupby('franja').head(n_top)

            # Guardar datos para exportar a KML
            df_kml_ready = df_obj.copy()

            # ---------------------------------------------------------
            # INTELIGENCIA: REDACCIÓN DE SINOPSIS
            # ---------------------------------------------------------
            def obtener_top1(df, franja_nombre):
                filtro = df[df['franja'] == franja_nombre]
                if not filtro.empty:
                    return f"Celda <b>{filtro.iloc[0]['celda']}</b> ({filtro.iloc[0]['nombre_lugar']}) con {filtro.iloc[0]['visitas']} conexiones."
                return "<i>Sin actividad registrada.</i>"

            noche_txt = obtener_top1(top_rutinas, '1. NOCHE (Pernocta: 22:00 - 02:59)')
            manana_txt = obtener_top1(top_rutinas, '2. MAÑANA/MADRUGADA (03:00 - 08:59)')
            dia_txt = obtener_top1(top_rutinas, '3. DÍA (Actividad: 09:00 - 21:59)')

            sinopsis = f"""
            <div style="background-color:#e8f4f8; padding:15px; border-left: 5px solid #0275d8; border-radius:5px; margin-bottom:15px;">
                <h4 style="margin-top:0; color:#0275d8;">💡 Sinopsis de Patrón de Vida</h4>
                <p>El análisis geoespacial segmentado del victimario <b>{num_vic}</b> arroja los siguientes lugares de alta recurrencia según la hora:</p>
                <ul>
                    <li><b>🌙 Lugar de Pernocta (Noche):</b> Su ubicación de descanso o refugio más probable es la {noche_txt}</li>
                    <li><b>🌅 Inicio de Jornada (Madrugada/Mañana):</b> Durante las primeras horas del día frecuenta principalmente la {manana_txt}</li>
                    <li><b>☀️ Zona de Operación (Día):</b> En horarios hábiles y comerciales, su principal área de movimiento es la {dia_txt}</li>
                </ul>
            </div>
            """
            display(HTML(sinopsis))

            # Mostrar tablas con estilo
            display(HTML(f"<h3>📊 Desglose de Rutinas ({drop_top.value})</h3>"))
            display(top_rutinas[['franja', 'celda', 'nombre_lugar', 'visitas', 'primer_registro', 'ultimo_registro']].style.background_gradient(cmap='Purples', subset=['visitas']).hide(axis='index'))

            # ---------------------------------------------------------
            # MAPA Y EXPORTACIÓN KML
            # ---------------------------------------------------------
            display(HTML("<hr><h3>🗺️ Visualización y Exportación 3D (Google Earth)</h3>"))

            df_mapa = top_rutinas.dropna(subset=['latitud', 'longitud'])
            if not df_mapa.empty:
                mapa = folium.Map(location=[df_mapa['latitud'].mean(), df_mapa['longitud'].mean()], zoom_start=13)

                # Colores por franja
                colores = {'1. NOCHE (Pernocta: 22:00 - 02:59)': 'darkblue',
                           '2. MAÑANA/MADRUGADA (03:00 - 08:59)': 'lightblue',
                           '3. DÍA (Actividad: 09:00 - 21:59)': 'orange'}

                for _, row in df_mapa.iterrows():
                    color = colores.get(row['franja'], 'gray')
                    html_pop = f"<b>{row['franja']}</b><br>Celda: {row['celda']}<br>Lugar: {row['nombre_lugar']}<br>Visitas: {row['visitas']}"
                    folium.Marker(
                        location=[row['latitud'], row['longitud']],
                        popup=folium.Popup(html_pop, max_width=250),
                        tooltip=f"{row['franja']} (Visitas: {row['visitas']})",
                        icon=folium.Icon(color=color, icon='info-sign')
                    ).add_to(mapa)

                display(mapa)

            btn_kml = widgets.Button(description='🌎 Descargar KML (Google Earth)', button_style='success', icon='globe', layout=widgets.Layout(width='250px'))
            out_kml = widgets.Output()

            def descargar_kml(b):
                with out_kml:
                    clear_output()
                    if df_kml_ready.empty: return

                    kml = simplekml.Kml(name=f"Rastreo_{num_vic}")

                    # Filtramos los que tienen lat/lon
                    datos_kml = df_kml_ready.dropna(subset=['latitud', 'longitud']).sort_values('fecha_limpia')

                    # Estilos de íconos en Google Earth
                    style_noche = simplekml.Style()
                    style_noche.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/paddle/blu-circle.png'
                    style_dia = simplekml.Style()
                    style_dia.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/paddle/ylw-circle.png'

                    for _, r in datos_kml.iterrows():
                        pnt = kml.newpoint(name=str(r['celda']), coords=[(r['longitud'], r['latitud'])])
                        pnt.description = f"<b>Fecha:</b> {r['fecha_limpia']}<br><b>Franja:</b> {r['franja']}"

                        if 'NOCHE' in r['franja']:
                            pnt.style = style_noche
                        else:
                            pnt.style = style_dia

                    arch_kml = f"Ruta_Victimario_{num_vic}.kml"
                    kml.save(arch_kml)
                    files.download(arch_kml)
                    print(f"✅ Archivo KML generado para Google Earth con {len(datos_kml)} puntos geo-referenciados.")

            btn_kml.on_click(descargar_kml)
            display(btn_kml)
            display(out_kml)

            del df_limpio
            gc.collect()

    btn_analizar.on_click(analizar_rutinas)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Button(button_style='primary', description=' Generar Perfil de Vida', icon='user-secret', layout=Layout(width=…

Output()